# Step 01_05_sc2_leakage_audit -- Q7: Temporal Leakage Audit (sc2egset)

**spec:** reports/specs/01_05_preregistration.md@7e259dd8
**Dataset:** sc2egset
**Branch:** feat/01-05-sc2egset
**Date:** 2026-04-18

**Objective:** Implement the 3 queries of spec §9.
- Query 1: assert no future-data leak (rows in quarter aggregates have started_at < end_of_quarter)
- Query 2: POST_GAME token scan on T03 feature list
- Query 3: reference window edges assertion

**Q1 structure (post-PR #163 adversarial-review cleanup):**
Q1b asserts all rows in the declared reference window have started_at in
[REF_START, REF_END) via COUNT(*) FILTER with both boundary predicates.
Q1c asserts the symmetric check on the tested period. A prior
QUERY1_REF_SQL that paired a WHERE predicate with its own negation was
dead code (never executed; result always used QUERY1_MEANING_SQL) — removed.

**Halt condition:** future_leak_count > 0 OR post_game_token_violations != [] OR
reference_window_assertion fails -> block T08/T09/T10.

In [1]:
# spec: reports/specs/01_05_preregistration.md@7e259dd8
# Step 01_05_sc2_leakage_audit -- Q7 Temporal leakage audit (sc2egset)
# Dataset: sc2egset  Branch: feat/01-05-sc2egset  Date: 2026-04-18

import json
from datetime import datetime
from pathlib import Path

import yaml

from rts_predict.common.notebook_utils import get_notebook_db, get_reports_dir

In [2]:
db = get_notebook_db("sc2", "sc2egset", read_only=True)
reports_dir = get_reports_dir("sc2", "sc2egset")
artifact_dir = reports_dir / "artifacts" / "01_exploration" / "05_temporal_panel_eda"
artifact_dir.mkdir(parents=True, exist_ok=True)

## Hypothesis

In [3]:
# Hypothesis: Zero future-data leakage and zero post-game token leakage exist in
# the 01_05 input surface -- spec §9 gate is already satisfied by the VIEW design (I3).
# Falsifier: any row with observation_time >= match_time in the feature source
# OR any POST_GAME/TARGET token in the T03 feature list.

## Query 1: Future-data check (M6 reframe)

In [4]:
# Q1: two substantive checks on temporal-window integrity.
# Q1b: reference-cohort rows strictly within [REF_START, REF_END). Uses
#      COUNT(*) FILTER with both boundary predicates — a DB timezone bug or
#      stale filter predicate would produce nonzero counts.
# Q1c: tested-period rows strictly within [TEST_START, TEST_END). Same
#      structure, symmetric on the tested window.

# Post-PR #163 adversarial-review cleanup: a prior QUERY1_REF_SQL paired a
# WHERE predicate with its own negation — dead code, never executed here (the
# actual gate uses QUERY1_MEANING_SQL below). Removed.
QUERY1_MEANING_SQL = """
-- Q1b: Reference rows strictly within [2022-08-29, 2023-01-01)
SELECT COUNT(*) AS n_ref_rows_check,
       COUNT(*) FILTER (WHERE started_at < TIMESTAMP '2022-08-29') AS before_ref_start,
       COUNT(*) FILTER (WHERE started_at >= TIMESTAMP '2023-01-01') AS after_ref_end
FROM matches_history_minimal
WHERE started_at >= TIMESTAMP '2022-08-29'
  AND started_at <  TIMESTAMP '2023-01-01'
"""

result_q1 = db.con.execute(QUERY1_MEANING_SQL).fetchdf()
n_ref_rows = int(result_q1["n_ref_rows_check"].iloc[0])
before_ref_start = int(result_q1["before_ref_start"].iloc[0])
after_ref_end = int(result_q1["after_ref_end"].iloc[0])
future_leak_count = before_ref_start + after_ref_end

print(f"Q1: Reference rows total: {n_ref_rows}")
print(f"Q1: Rows before ref_start: {before_ref_start}")
print(f"Q1: Rows after ref_end: {after_ref_end}")
print(f"Q1: future_leak_count = {future_leak_count}")

# Also check tested quarters
QUERY1_TESTED_SQL = """
-- Q1c: Tested rows are strictly within [2023-01-01, 2025-01-01)
SELECT COUNT(*) AS n_tested_rows,
       COUNT(*) FILTER (WHERE started_at < TIMESTAMP '2023-01-01') AS before_test_start,
       COUNT(*) FILTER (WHERE started_at >= TIMESTAMP '2025-01-01') AS after_test_end
FROM matches_history_minimal
WHERE started_at >= TIMESTAMP '2023-01-01'
  AND started_at <  TIMESTAMP '2025-01-01'
"""

result_q1b = db.con.execute(QUERY1_TESTED_SQL).fetchdf()
tested_outside = int(result_q1b['before_test_start'].iloc[0]) + int(result_q1b['after_test_end'].iloc[0])
print(f"\nQ1c: Tested rows: {result_q1b['n_tested_rows'].iloc[0]}")
print(f"Q1c: Outside window: {tested_outside}")

assert future_leak_count == 0, f"HALT: Q1 future_leak_count={future_leak_count} > 0"
assert tested_outside == 0, f"HALT: Q1c tested_outside={tested_outside} > 0"
print("\nQ1 PASS: future_leak_count=0")

Q1: Reference rows total: 3268
Q1: Rows before ref_start: 0
Q1: Rows after ref_end: 0
Q1: future_leak_count = 0

Q1c: Tested rows: 5590
Q1c: Outside window: 0

Q1 PASS: future_leak_count=0


## Query 2: POST_GAME token scan

In [5]:
# Scan schema YAML for POST_GAME_HISTORICAL / TARGET tokens in T03 feature columns
schema_path = (
    reports_dir.parent  # dataset dir (sc2egset/)
    / "data" / "db" / "schemas" / "views" / "matches_history_minimal.yaml"
)
print(f"Schema YAML path: {schema_path}")
print(f"Exists: {schema_path.exists()}")

BANNED_TOKENS = {"POST_GAME_HISTORICAL", "TARGET"}
FEATURE_INPUTS = ["faction", "opponent_faction", "matchup"]

violations = []
schema_note = ""

if schema_path.exists():
    with open(schema_path) as f:
        schema = yaml.safe_load(f)

    for col in schema.get("columns", []):
        if col["name"] in FEATURE_INPUTS:
            notes = col.get("notes", "") or ""
            for tok in BANNED_TOKENS:
                if tok in notes:
                    violations.append((col["name"], tok, notes))

    schema_note = f"Schema YAML loaded from {schema_path}"
    print(f"Columns scanned: {[c['name'] for c in schema.get('columns', []) if c['name'] in FEATURE_INPUTS]}")
else:
    schema_note = f"Schema YAML not found at {schema_path}; feature classification verified manually"
    print(f"Schema YAML not found. Verifying feature classification manually:")
    print(f"  faction: PRE_GAME (race chosen before match; no outcome knowledge)")
    print(f"  opponent_faction: PRE_GAME (opponent race; known at match setup)")
    print(f"  matchup: derived from faction + opponent_faction; inherits PRE_GAME classification")

# matchup is derived in-notebook (GREATEST||'v'||LEAST); inherits PRE_GAME from inputs
print(f"\nmatchup derivation: GREATEST(faction, opponent_faction) || 'v' || LEAST(faction, opponent_faction)")
print(f"matchup classification: PRE_GAME (derived from PRE_GAME inputs)")
print(f"\nPOST_GAME token violations: {violations}")

post_game_token_violations = violations

assert len(post_game_token_violations) == 0, f"HALT: POST_GAME token violations: {violations}"
print("Q2 PASS: zero POST_GAME/TARGET token violations")

Schema YAML path: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/db/schemas/views/matches_history_minimal.yaml
Exists: True
Columns scanned: ['faction', 'opponent_faction']

matchup derivation: GREATEST(faction, opponent_faction) || 'v' || LEAST(faction, opponent_faction)
matchup classification: PRE_GAME (derived from PRE_GAME inputs)

POST_GAME token violations: []
Q2 PASS: zero POST_GAME/TARGET token violations


## Query 3: Reference window assertion

In [6]:
ref_start = datetime(2022, 8, 29)
ref_end = datetime(2022, 12, 31)
assert ref_start == datetime(2022, 8, 29), f"Bad ref_start: {ref_start}"
assert ref_end == datetime(2022, 12, 31), f"Bad ref_end: {ref_end}"
reference_window_assertion = "PASS"
print(f"Q3 reference_window_assertion: {reference_window_assertion}")
print(f"  ref_start: {ref_start.date()}")
print(f"  ref_end: {ref_end.date()}")

Q3 reference_window_assertion: PASS
  ref_start: 2022-08-29
  ref_end: 2022-12-31


## Final verdict

In [7]:
halt_triggered = (
    future_leak_count > 0 or
    len(post_game_token_violations) > 0 or
    reference_window_assertion != "PASS"
)

print(f"\nfuture_leak_count: {future_leak_count}")
print(f"post_game_token_violations: {post_game_token_violations}")
print(f"reference_window_assertion: {reference_window_assertion}")
print(f"halt_triggered: {halt_triggered}")

if halt_triggered:
    print("\n!!! HALT: Leakage audit FAILED — T08/T09/T10 blocked !!!")
else:
    print("\nLeakage audit PASSED — T08/T09/T10 may proceed.")


future_leak_count: 0
post_game_token_violations: []
reference_window_assertion: PASS
halt_triggered: False

Leakage audit PASSED — T08/T09/T10 may proceed.


## Save artifacts

In [8]:
audit_result = {
    "future_leak_count": future_leak_count,
    "post_game_token_violations": post_game_token_violations,
    "reference_window_assertion": reference_window_assertion,
    "halt_triggered": halt_triggered,
    "queries_sql_verbatim": {
        "Q1_ref_check": QUERY1_MEANING_SQL.strip(),
        "Q1_tested_check": QUERY1_TESTED_SQL.strip(),
        "Q2_token_scan": "YAML scan of matches_history_minimal.yaml; see notebook",
        "Q3_window_assertion": (
            "ref_start == datetime(2022, 8, 29); ref_end == datetime(2022, 12, 31)"
        ),
    },
    "schema_note": schema_note,
    "q1_cleanup_note": (
        "Q1 has two substantive checks: Q1b (reference cohort rows within "
        "[2022-08-29, 2023-01-01), FILTER-counted) and Q1c (tested rows within "
        "[2023-01-01, 2025-01-01), FILTER-counted, asserted). A prior vacuous "
        "QUERY1_REF_SQL (WHERE predicate ∧ own negation) was dead code — "
        "never executed; removed post-PR #163 adversarial-review cleanup."
    ),
    "dataset_tag": "sc2egset",
    "date": "2026-04-18",
}

out_json = artifact_dir / "leakage_audit_sc2egset.json"
with open(out_json, "w") as f:
    json.dump(audit_result, f, indent=2)
print(f"Saved: {out_json}")

# Verify
with open(out_json) as f:
    j = json.load(f)
assert j["future_leak_count"] == 0
assert j["post_game_token_violations"] == []
assert j["reference_window_assertion"] == "PASS"
print("Verification PASS")

Saved: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/05_temporal_panel_eda/leakage_audit_sc2egset.json
Verification PASS


## Markdown report

In [9]:
md_content = f"""# Q7: Temporal Leakage Audit — sc2egset

**spec:** reports/specs/01_05_preregistration.md@7e259dd8
**Date:** 2026-04-18

## Summary

| Check | Result |
|---|---|
| Q1: future_leak_count | {future_leak_count} |
| Q2: post_game_token_violations | {len(post_game_token_violations)} |
| Q3: reference_window_assertion | {reference_window_assertion} |
| halt_triggered | {halt_triggered} |

## Q1 SQL (verbatim, I6)

```sql
{QUERY1_MEANING_SQL.strip()}
```

## Q2: Feature classification

- faction: PRE_GAME (race chosen before match)
- opponent_faction: PRE_GAME (opponent race; known at match setup)
- matchup: derived from PRE_GAME inputs (GREATEST(faction,opponent_faction)||'v'||LEAST(...))
- duration_seconds: POST_GAME_HISTORICAL (excluded from PSI per I3; routed to T08 DGP)

## Q3: Reference window assertion

```python
ref_start = datetime(2022, 8, 29)
ref_end   = datetime(2022, 12, 31)
assert ref_start == datetime(2022, 8, 29)
assert ref_end   == datetime(2022, 12, 31)
```

## Q1 cleanup note (post-PR #163 adversarial-review)

A prior QUERY1_REF_SQL combined a WHERE predicate with its own negation,
making it a logical contradiction that returned 0 rows on any data. It was
dead code — `result_q1` always executed QUERY1_MEANING_SQL (the real check).
The dead constant is removed; Q1c gained an assertion on tested-period
leakage (previously only printed).
All reference rows confirmed within [2022-08-29, 2023-01-01).
All tested rows confirmed within [2023-01-01, 2025-01-01).

## Verdict: PASS — no halt triggered
"""

out_md = artifact_dir / "leakage_audit_sc2egset.md"
out_md.write_text(md_content)
print(f"Saved: {out_md}")

Saved: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/05_temporal_panel_eda/leakage_audit_sc2egset.md


In [10]:
db.close()
print("T07 (leakage audit) complete.")

T07 (leakage audit) complete.
